<a href="https://colab.research.google.com/github/mohamedelguendy/library-management-system-python/blob/main/library_management_system_python.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [37]:
from google.colab import drive
drive.mount('/content/drive')

BOOKS_FILE = "/content/drive/MyDrive/Library Management System/books.json"
USERS_FILE = "/content/drive/MyDrive/Library Management System/users.json"
FINES_FILE = "/content/drive/MyDrive/Library Management System/fines.json"
HISTORY_FILE = "/content/drive/MyDrive/Library Management System/history.json"
STAFF_FILE  = "/content/drive/MyDrive/Library Management System/staff.json"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [39]:
from google.colab import drive
drive.mount('/content/drive')

import datetime
import logging
import json
import os
import hashlib
import getpass

# ==============================
# LOGGING
# ==============================
logging.basicConfig(level=logging.INFO, format="%(levelname)s: %(message)s")

BASE_DIR = "/content/drive/MyDrive/Library Management System"

# Shared across all libraries
USERS_FILE   = f"{BASE_DIR}/users.json"
FINES_FILE   = f"{BASE_DIR}/fines.json"
HISTORY_FILE = f"{BASE_DIR}/history.json"
STAFF_FILE   = f"{BASE_DIR}/staff.json"

# Per-library book files
def books_file(lib_name):
    safe = lib_name.replace(" ", "_")
    return f"{BASE_DIR}/books_{safe}.json"

# ==============================
# PASSWORD HELPERS
# ==============================
def hash_password(password: str) -> str:
    return hashlib.sha256(password.encode()).hexdigest()

def get_hidden_password(prompt="Password: ") -> str:
    return getpass.getpass(prompt)

# ==============================
# CUSTOM EXCEPTIONS
# ==============================
class BookNotFoundError(Exception): pass
class UserNotFoundError(Exception): pass
class BorrowLimitError(Exception): pass
class BookUnavailableError(Exception): pass
class AuthError(Exception): pass

# ==============================
# BOOK CLASS
# ==============================
class Book:
    def __init__(self, name, author, copies):
        if not name or not author or copies < 0:
            raise ValueError("Invalid book data")
        self.name   = name
        self.author = author
        self.copies = copies

    def to_dict(self):
        return {"name": self.name, "author": self.author, "copies": self.copies}

# ==============================
# USER REGISTRY  (shared, singleton)
# ==============================
class UserRegistry:
    """
    One registry for ALL libraries.
    Stores users, their fines, and their full cross-library history.
    """
    _instance = None

    @classmethod
    def get(cls):
        if cls._instance is None:
            cls._instance = UserRegistry()
        return cls._instance

    def __init__(self):
        self.users = {}   # user_id -> User
        self._load_users()
        self._load_fines()
        self._load_history()

    # ---- persistence ----
    def _load_users(self):
        if not os.path.exists(USERS_FILE): return
        with open(USERS_FILE, "r") as f:
            data = json.load(f)
        for uid, info in data.items():
            user = User(
                info["user_id"], info["name"],
                info.get("password_hash", ""),
                info.get("security_question", ""),
                info.get("security_answer_hash", ""),
            )
            user.borrowed_books = info.get("borrowed_books", {})
            user.fine           = info.get("fine", 0)
            user.history        = info.get("history", [])
            self.users[uid]     = user

    def _load_fines(self):
        if not os.path.exists(FINES_FILE): return
        with open(FINES_FILE, "r") as f:
            data = json.load(f)
        for uid, info in data.items():
            if uid in self.users:
                self.users[uid].fine = info["fine"]

    def _load_history(self):
        if not os.path.exists(HISTORY_FILE): return
        with open(HISTORY_FILE, "r") as f:
            data = json.load(f)
        for uid, history in data.items():
            if uid in self.users:
                self.users[uid].history = history

    def save_users(self):
        os.makedirs(BASE_DIR, exist_ok=True)
        with open(USERS_FILE, "w") as f:
            json.dump({uid: u.to_dict() for uid, u in self.users.items()}, f, indent=4)

    def save_fines(self):
        os.makedirs(BASE_DIR, exist_ok=True)
        with open(FINES_FILE, "w") as f:
            json.dump({uid: {"name": u.name, "fine": u.fine} for uid, u in self.users.items()}, f, indent=4)

    def save_history(self):
        os.makedirs(BASE_DIR, exist_ok=True)
        with open(HISTORY_FILE, "w") as f:
            json.dump({uid: u.history for uid, u in self.users.items()}, f, indent=4)

    def save_all(self):
        self.save_users()
        self.save_fines()
        self.save_history()

    # ---- user ops ----
    def add_user(self, user_id, name):
        if user_id in self.users:
            raise Exception("User already exists")
        print(f"\n--- Set password for new user '{name}' ---")
        password = get_hidden_password("Password: ")
        confirm  = get_hidden_password("Confirm password: ")
        if password != confirm:
            raise AuthError("Passwords do not match.")
        question = input("Security question (for forgot-password): ").strip()
        answer   = get_hidden_password("Security answer: ").strip().lower()
        self.users[user_id] = User(
            user_id, name,
            hash_password(password),
            question,
            hash_password(answer),
        )
        self.save_all()
        logging.info(f"[Registry] User added: {user_id}")
        print(f"User '{name}' registered successfully.")

    def remove_user(self, user_id):
        if user_id not in self.users:
            raise UserNotFoundError("User not found")
        del self.users[user_id]
        self.save_all()

    def login(self) -> "User":
        uid = input("User ID: ").strip()
        if uid not in self.users:
            raise UserNotFoundError("User not found.")
        password = get_hidden_password("User password: ")
        if not self.users[uid].verify_password(password):
            raise AuthError("Incorrect password.")
        print(f"Welcome, {self.users[uid].name}!")
        return self.users[uid]

    def forgot_password(self):
        uid = input("User ID: ").strip()
        if uid not in self.users:
            raise UserNotFoundError("User not found.")
        user = self.users[uid]
        if not user.security_question:
            raise AuthError("No security question set for this user.")
        print(f"Security question: {user.security_question}")
        answer = get_hidden_password("Your answer: ").strip().lower()
        if hash_password(answer) != user.security_answer_hash:
            raise AuthError("Incorrect answer.")
        new_pw  = get_hidden_password("New password: ")
        confirm = get_hidden_password("Confirm new password: ")
        if new_pw != confirm:
            raise AuthError("Passwords do not match.")
        user.password_hash = hash_password(new_pw)
        self.save_users()
        print(f"Password reset successfully for '{user.name}'.")

# ==============================
# USER CLASS
# ==============================
class User:
    def __init__(self, user_id, name, password_hash="",
                 security_question="", security_answer_hash=""):
        if not user_id or not name:
            raise ValueError("Invalid user data")
        self.user_id              = user_id
        self.name                 = name
        self.password_hash        = password_hash or hash_password("1234")
        self.security_question    = security_question
        self.security_answer_hash = security_answer_hash
        # borrowed_books: { "Library Name::Book Name": {borrow_date, due_date} }
        self.borrowed_books       = {}
        self.fine                 = 0
        # history entries include "library" field
        self.history              = []

    def total_borrowed(self):
        return len(self.borrowed_books)

    def can_borrow(self):
        return self.total_borrowed() < 3 and self.fine == 0

    def verify_password(self, password: str) -> bool:
        return self.password_hash == hash_password(password)

    def to_dict(self):
        return {
            "user_id":               self.user_id,
            "name":                  self.name,
            "password_hash":         self.password_hash,
            "security_question":     self.security_question,
            "security_answer_hash":  self.security_answer_hash,
            "borrowed_books":        self.borrowed_books,
            "fine":                  self.fine,
            "history":               self.history,
        }

# ==============================
# STAFF MANAGER
# ==============================
class StaffManager:
    def __init__(self):
        self.staff = {}
        self._load()
        if not self.staff:
            self._seed_default_admin()

    def _load(self):
        if not os.path.exists(STAFF_FILE): return
        with open(STAFF_FILE, "r") as f:
            self.staff = json.load(f)

    def _save(self):
        os.makedirs(BASE_DIR, exist_ok=True)
        with open(STAFF_FILE, "w") as f:
            json.dump(self.staff, f, indent=4)

    def _seed_default_admin(self):
        self.staff["admin"] = {
            "password_hash":        hash_password("admin123"),
            "security_question":    "What is the library name?",
            "security_answer_hash": hash_password("pahros"),
        }
        self._save()
        print("[Staff] Default admin created  →  username: admin | password: admin123")

    def login(self) -> str:
        username = input("Staff username: ").strip()
        if username not in self.staff:
            raise AuthError("Staff account not found.")
        password = get_hidden_password("Staff password: ")
        if hash_password(password) != self.staff[username]["password_hash"]:
            raise AuthError("Incorrect password.")
        print(f"[Staff] Welcome, {username}!")
        return username

    def forgot_password(self):
        username = input("Staff username: ").strip()
        if username not in self.staff:
            raise AuthError("Staff account not found.")
        record = self.staff[username]
        print(f"Security question: {record['security_question']}")
        answer = get_hidden_password("Your answer: ").strip().lower()
        if hash_password(answer) != record["security_answer_hash"]:
            raise AuthError("Incorrect answer.")
        new_pw  = get_hidden_password("New password: ")
        confirm = get_hidden_password("Confirm new password: ")
        if new_pw != confirm:
            raise AuthError("Passwords do not match.")
        record["password_hash"] = hash_password(new_pw)
        self._save()
        print("[Staff] Password reset successfully.")

    def register(self):
        username = input("New staff username: ").strip()
        if username in self.staff:
            raise Exception("Username already exists.")
        password = get_hidden_password("Password: ")
        confirm  = get_hidden_password("Confirm password: ")
        if password != confirm:
            raise AuthError("Passwords do not match.")
        question = input("Security question: ").strip()
        answer   = get_hidden_password("Security answer: ").strip().lower()
        self.staff[username] = {
            "password_hash":        hash_password(password),
            "security_question":    question,
            "security_answer_hash": hash_password(answer),
        }
        self._save()
        print(f"[Staff] Account '{username}' created.")

# ==============================
# FINE CALCULATOR
# ==============================
class FineCalculator:
    @staticmethod
    def calculate(days_late):
        return 0 if days_late <= 0 else days_late * 5

# ==============================
# LIBRARY CLASS  (books only — users are in the registry)
# ==============================
class Library:
    def __init__(self, library_name):
        self.library_name = library_name
        self.books        = {}
        self.registry     = UserRegistry.get()
        self._load_books()

    def log(self, message):
        logging.info(f"[{self.library_name}] {message}")

    def _load_books(self):
        path = books_file(self.library_name)
        if not os.path.exists(path): return
        with open(path, "r") as f:
            data = json.load(f)
        for name, info in data.items():
            self.books[name] = Book(info["name"], info["author"], info["copies"])

    def _save_books(self):
        os.makedirs(BASE_DIR, exist_ok=True)
        path = books_file(self.library_name)
        with open(path, "w") as f:
            json.dump({name: b.to_dict() for name, b in self.books.items()}, f, indent=4)

    # ---- book ops ----
    def add_book(self, name, author, copies):
        if name in self.books:
            self.books[name].copies += copies
        else:
            self.books[name] = Book(name, author, copies)
        self._save_books()
        self.log(f"Book added: {name}")

    def remove_book(self, name):
        if name not in self.books:
            raise BookNotFoundError("Book not found")
        del self.books[name]
        self._save_books()
        self.log(f"Book removed: {name}")

    def show_books(self):
        print(f"\n[{self.library_name}] Books:")
        for b in self.books.values():
            print(f"  {b.name} | {b.author} | Copies: {b.copies}")

    def show_users(self):
        print(f"\n[{self.library_name}] All registered users:")
        for u in self.registry.users.values():
            print(f"  {u.user_id} | {u.name} | Fine: {u.fine} EGP | Borrowed: {u.total_borrowed()}")

    # ---- borrow ----
    def borrow_book(self, user: User):
        if not user.can_borrow():
            raise BorrowLimitError(
                f"Cannot borrow: {'outstanding fine' if user.fine > 0 else 'limit of 3 books reached (across all libraries)'}"
            )
        print(f"\n[{self.library_name}] WARNING: 5 EGP/day fine, 7-day limit")
        book_name = input("Enter book name: ").strip()
        if book_name not in self.books:
            raise BookNotFoundError("Book not found in this library")
        book = self.books[book_name]
        if book.copies <= 0:
            raise BookUnavailableError("No copies available")

        # Key includes library so same-named books from diff libraries don't clash
        key         = f"{self.library_name}::{book_name}"
        borrow_date = datetime.datetime.now()
        due_date    = borrow_date + datetime.timedelta(days=7)

        user.borrowed_books[key] = {
            "library":     self.library_name,
            "book":        book_name,
            "borrow_date": borrow_date.isoformat(),
            "due_date":    due_date.isoformat(),
        }
        user.history.append({
            "library":     self.library_name,
            "book":        book_name,
            "borrow_time": borrow_date.isoformat(),
            "return_time": None,
            "fine":        0,
        })
        book.copies -= 1

        self._save_books()
        self.registry.save_all()
        self.log(f"{user.user_id} borrowed '{book_name}'")
        print(f"  Borrowed '{book_name}' from {self.library_name}. Due: {due_date.strftime('%Y-%m-%d')}")

    # ---- return ----
    def return_book(self, user_id, book_name):
        if user_id not in self.registry.users:
            raise UserNotFoundError("User not found")
        user = self.registry.users[user_id]
        key  = f"{self.library_name}::{book_name}"
        if key not in user.borrowed_books:
            raise BookNotFoundError(f"'{book_name}' not borrowed from {self.library_name}")

        due_date    = datetime.datetime.fromisoformat(user.borrowed_books[key]["due_date"])
        return_date = datetime.datetime.now()
        days_late   = (return_date - due_date).days
        fine        = FineCalculator.calculate(days_late)
        user.fine  += fine

        del user.borrowed_books[key]
        self.books[book_name].copies += 1

        for record in user.history:
            if (record["library"] == self.library_name and
                record["book"] == book_name and
                record["return_time"] is None):
                record["return_time"] = return_date.isoformat()
                record["fine"]        = fine
                break

        self._save_books()
        self.registry.save_all()
        self.log(f"{user_id} returned '{book_name}' | fine: {fine} EGP")
        return fine

# ==============================
# SETUP LIBRARIES
# ==============================
def create_libraries():
    lib1 = Library("Pahros Library")
    lib1.add_book("Python Basics", "Ahmed Hassan", 5)
    lib1.add_book("AI Intro", "John Smith", 3)

    lib2 = Library("Alex Library")
    lib2.add_book("Data Science", "Ali Mohamed", 4)
    lib2.add_book("ML Guide", "Sara Ibrahim", 2)

    lib3 = Library("Digital Library")
    lib3.add_book("Cyber Security", "Hassan Ali", 6)

    return {"1": lib1, "2": lib2, "3": lib3}

# ==============================
# PORTALS
# ==============================
def user_portal(lib):
    registry = UserRegistry.get()
    user = registry.login()

    while True:
        print(f"\n  [{user.name}'s Portal]")
        print(f"  Currently at: {lib.library_name}")
        print("  1. Borrow a Book (from this library)")
        print("  2. My Borrowed Books (all libraries)")
        print("  3. My Fine")
        print("  4. My Full History (all libraries)")
        print("  5. Change My Password")
        print("  6. Logout")

        uc = input("  Choice: ").strip()
        try:
            if uc == "1":
                lib.borrow_book(user)

            elif uc == "2":
                if not user.borrowed_books:
                    print("  You have no borrowed books.")
                else:
                    print(f"\n  All borrowed books for {user.name}:")
                    for key, info in user.borrowed_books.items():
                        due = info["due_date"][:10]
                        print(f"    - [{info['library']}]  {info['book']}  |  Due: {due}")

            elif uc == "3":
                print(f"\n  Outstanding fine: {user.fine} EGP")

            elif uc == "4":
                if not user.history:
                    print("  No history yet.")
                else:
                    print(f"\n  Full history for {user.name}:")
                    for i, rec in enumerate(user.history, 1):
                        returned = rec["return_time"][:10] if rec["return_time"] else "Not returned yet"
                        print(f"    {i}. [{rec['library']}] {rec['book']} | "
                              f"Borrowed: {rec['borrow_time'][:10]} | "
                              f"Returned: {returned} | Fine: {rec['fine']} EGP")

            elif uc == "5":
                old_pw = get_hidden_password("  Current password: ")
                if not user.verify_password(old_pw):
                    print("  Incorrect current password.")
                else:
                    new_pw  = get_hidden_password("  New password: ")
                    confirm = get_hidden_password("  Confirm new password: ")
                    if new_pw != confirm:
                        print("  Passwords do not match.")
                    else:
                        user.password_hash = hash_password(new_pw)
                        registry.save_users()
                        print("  Password changed successfully.")

            elif uc == "6":
                print(f"  Goodbye, {user.name}!")
                break
            else:
                print("  Invalid choice.")

        except Exception as e:
            print("  Error:", e)


def staff_portal(lib, staff_mgr):
    registry = UserRegistry.get()
    while True:
        print(f"\n  [Staff Portal – {lib.library_name}]")
        print("  1. Show All Books (this library)")
        print("  2. Add Book")
        print("  3. Remove Book")
        print("  4. Add User (global)")
        print("  5. Remove User (global)")
        print("  6. Show All Users")
        print("  7. Return Book (on behalf of user)")
        print("  8. Register New Staff Account")
        print("  9. Logout")

        sc = input("  Choice: ").strip()
        try:
            if sc == "1":
                lib.show_books()

            elif sc == "2":
                n = input("  Book name: ").strip()
                a = input("  Author: ").strip()
                c = int(input("  Copies: ").strip())
                lib.add_book(n, a, c)

            elif sc == "3":
                n = input("  Book name to remove: ").strip()
                lib.remove_book(n)

            elif sc == "4":
                uid  = input("  User ID: ").strip()
                name = input("  Name: ").strip()
                registry.add_user(uid, name)

            elif sc == "5":
                uid = input("  User ID to remove: ").strip()
                registry.remove_user(uid)
                print("  User removed.")

            elif sc == "6":
                lib.show_users()

            elif sc == "7":
                uid  = input("  User ID: ").strip()
                book = input("  Book name: ").strip()
                fine = lib.return_book(uid, book)
                print(f"  Returned. Fine: {fine} EGP")

            elif sc == "8":
                staff_mgr.register()

            elif sc == "9":
                print("  [Staff] Logged out.")
                break
            else:
                print("  Invalid choice.")

        except Exception as e:
            print("  Error:", e)


def forgot_password_menu(lib, staff_mgr):
    registry = UserRegistry.get()
    print("\n  === Forgot Password ===")
    print("  1. Staff")
    print("  2. User")
    fp = input("  Choice: ").strip()
    if fp == "1":
        staff_mgr.forgot_password()
    elif fp == "2":
        registry.forgot_password()
    else:
        print("  Invalid choice.")

# ==============================
# MAIN
# ==============================
staff_mgr = StaffManager()
registry  = UserRegistry.get()
libraries = create_libraries()

while True:
    print("\n===== SELECT LIBRARY =====")
    print("1. Pahros Library")
    print("2. Alex Library")
    print("3. Digital Library")
    print("4. Exit")

    choice = input("Choose library: ").strip()
    if choice == "4":
        print("System closed.")
        break
    if choice not in libraries:
        print("Invalid choice.")
        continue

    lib = libraries[choice]
    print(f"\nWelcome to {lib.library_name}")

    while True:
        print(f"\n[{lib.library_name}] MENU")
        print("1. Show Books")
        print("2. Return Book")
        print("3. User Portal  (Login)")
        print("4. Staff Portal (Login)")
        print("5. Forgot Password")
        print("6. Back to Library Select")

        c = input("Choice: ").strip()
        try:
            if c == "1":
                lib.show_books()
            elif c == "2":
                uid  = input("User ID: ").strip()
                book = input("Book name: ").strip()
                fine = lib.return_book(uid, book)
                print(f"Returned. Fine: {fine} EGP")
            elif c == "3":
                user_portal(lib)
            elif c == "4":
                staff_mgr.login()
                staff_portal(lib, staff_mgr)
            elif c == "5":
                forgot_password_menu(lib, staff_mgr)
            elif c == "6":
                break
            else:
                print("Invalid choice.")

        except Exception as e:
            print("Error:", e)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

===== SELECT LIBRARY =====
1. Pahros Library
2. Alex Library
3. Digital Library
4. Exit
Choose library: 1

Welcome to Pahros Library

[Pahros Library] MENU
1. Show Books
2. Return Book
3. User Portal  (Login)
4. Staff Portal (Login)
5. Forgot Password
6. Back to Library Select
Choice: 6

===== SELECT LIBRARY =====
1. Pahros Library
2. Alex Library
3. Digital Library
4. Exit
Choose library: 4
System closed.
